In [1]:
import pandas as pd

# Load CSV directly from Kaggle dataset
df = pd.read_csv("/kaggle/input/common-voice/cv-valid-train.csv")

print(df.shape)
df.tail()

(195776, 8)


,filename,text,up_votes,down_votes,age,gender,accent,duration
195771,cv-valid-train/sample-195771.mp3,the englishman said nothing,1,0,thirties,male,england,NaN
195772,cv-valid-train/sample-195772.mp3,the irish man sipped his tea,1,0,NaN,NaN,NaN,NaN
195773,cv-valid-train/sample-195773.mp3,what do you know about that,1,0,NaN,NaN,NaN,NaN
195774,cv-valid-train/sample-195774.mp3,the phone rang while she was awake,2,0,twenties,male,us,NaN
195775,cv-valid-train/sample-195775.mp3,among these people were a couple of cyclists a...,4,1,NaN,NaN,NaN,NaN


In [2]:
df_filtered = df.dropna(subset=['age', 'gender'])
print("After age & gender filter:", df_filtered.shape[0])

After age & gender filter: 73466


In [3]:
df_filtered = df_filtered[df_filtered['gender'].isin(['male', 'female'])]
print("After removing 'other' gender:", df_filtered.shape[0])
df_filtered['gender'].value_counts()

After removing 'other' gender: 72692


gender
male      54593
female    18099
Name: count, dtype: int64

In [5]:
age_map = {
    'teens': 15,
    'twenties': 25,
    'thirties': 35,
    'fourties': 45,
    'fifties': 55,
    'sixties': 65,
    'seventies': 75,
    'eighties': 85,
    'nineties': 95
}

df_filtered['age_num'] = df_filtered['age'].map(age_map)
df_filtered = df_filtered.dropna(subset=['age_num'])

# Age class: 0 = below 60, 1 = senior citizen
df_filtered['age_class'] = (df_filtered['age_num'] >= 60).astype(int)

df_filtered[['age', 'age_num', 'age_class']].head()


,age,age_num,age_class
5,twenties,25,0
8,seventies,75,1
13,thirties,35,0
14,sixties,65,1
19,fifties,55,0


In [6]:
SAMPLES_PER_GROUP = 2500  # total = 10,000

df_sampled = (
    df_filtered
    .groupby(['gender', 'age_class'], group_keys=False)
    .apply(lambda x: x.sample(min(len(x), SAMPLES_PER_GROUP), random_state=42))
)

print("Final sampled dataset:", df_sampled.shape)
df_sampled[['gender', 'age_class']].value_counts()


Final sampled dataset: (9370, 10)


/tmp/ipykernel_55/1431388835.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), SAMPLES_PER_GROUP), random_state=42))


gender  age_class
female  0            2500
male    0            2500
        1            2500
female  1            1870
Name: count, dtype: int64

In [7]:
df_sampled = df_sampled[['filename', 'gender', 'age_class']]
df_sampled.head()

,filename,gender,age_class
82168,cv-valid-train/sample-082168.mp3,female,0
90601,cv-valid-train/sample-090601.mp3,female,0
20257,cv-valid-train/sample-020257.mp3,female,0
10628,cv-valid-train/sample-010628.mp3,female,0
137881,cv-valid-train/sample-137881.mp3,female,0


In [8]:
gender_map = {'male': 0, 'female': 1}
df_filtered['gender_class'] = df_filtered['gender'].map(gender_map)

In [9]:
df_final = df_filtered[['filename', 'gender_class', 'age_class']]
df_final.head()

,filename,gender_class,age_class
5,cv-valid-train/sample-000005.mp3,1,0
8,cv-valid-train/sample-000008.mp3,0,1
13,cv-valid-train/sample-000013.mp3,1,0
14,cv-valid-train/sample-000014.mp3,0,1
19,cv-valid-train/sample-000019.mp3,0,0


In [11]:
SAMPLES_PER_GENDER = 4000

df_sampled = (
    df_final
    .groupby('gender_class', group_keys=False)
    .apply(lambda x: x.sample(min(len(x), SAMPLES_PER_GENDER), random_state=42))
)

print("Final dataset shape:", df_sampled.shape)
df_sampled['gender_class'].value_counts()

Final dataset shape: (8000, 3)


/tmp/ipykernel_55/1462099126.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(min(len(x), SAMPLES_PER_GENDER), random_state=42))


gender_class
0    4000
1    4000
Name: count, dtype: int64

In [13]:
import os
import numpy as np
import librosa
import pandas as pd
from tqdm import tqdm

In [14]:
AUDIO_DIR = "/kaggle/input/common-voice/cv-valid-train/"

df_sampled['audio_path'] = df_sampled['filename'].apply(
    lambda x: os.path.join(AUDIO_DIR, x)
)

df_sampled[['audio_path']].head()


,audio_path
140482,/kaggle/input/common-voice/cv-valid-train/cv-v...
101187,/kaggle/input/common-voice/cv-valid-train/cv-v...
82098,/kaggle/input/common-voice/cv-valid-train/cv-v...
141291,/kaggle/input/common-voice/cv-valid-train/cv-v...
132524,/kaggle/input/common-voice/cv-valid-train/cv-v...


In [15]:
def extract_mfcc(file_path, n_mfcc=40, max_len=174):
    try:
        audio, sr = librosa.load(file_path, sr=16000, duration=3)
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=n_mfcc)
        
        # Padding or truncation
        if mfcc.shape[1] < max_len:
            pad_width = max_len - mfcc.shape[1]
            mfcc = np.pad(mfcc, pad_width=((0,0),(0,pad_width)), mode='constant')
        else:
            mfcc = mfcc[:, :max_len]
            
        return mfcc
    except Exception as e:
        return None


In [16]:
X = []
y_gender = []
y_age = []

In [17]:
CHUNK_SIZE = 1000

for start in range(0, len(df_sampled), CHUNK_SIZE):
    end = start + CHUNK_SIZE
    chunk = df_sampled.iloc[start:end]
    
    for _, row in tqdm(chunk.iterrows(), total=len(chunk)):
        mfcc = extract_mfcc(row['audio_path'])
        
        if mfcc is not None:
            X.append(mfcc)
            y_gender.append(row['gender_class'])
            y_age.append(row['age_class'])
    
    print(f"Processed {len(X)} samples so far")

100%|██████████| 1000/1000 [00:40<00:00, 24.85it/s]


Processed 1000 samples so far


100%|██████████| 1000/1000 [00:27<00:00, 36.83it/s]


Processed 2000 samples so far


100%|██████████| 1000/1000 [00:28<00:00, 35.42it/s]


...

Processed 6000 samples so far


100%|██████████| 1000/1000 [00:23<00:00, 42.25it/s]


Processed 7000 samples so far


100%|██████████| 1000/1000 [00:23<00:00, 42.23it/s]

Processed 8000 samples so far


In [18]:
X = np.array(X)
y_gender = np.array(y_gender)
y_age = np.array(y_age)

X = X[..., np.newaxis]  # add channel dimension

print("X shape:", X.shape)
print("Gender labels:", y_gender.shape)
print("Age labels:", y_age.shape)

X shape: (8000, 40, 174, 1)
Gender labels: (8000,)
Age labels: (8000,)


In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_gender_train, y_gender_test, y_age_train, y_age_test = train_test_split(
    X, y_gender, y_age, test_size=0.2, random_state=42
)

In [21]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

gender_model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=X_train.shape[1:]),
    MaxPooling2D((2,2)),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

gender_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

gender_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
I0000 00:00:1768400682.563043      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1768400682.566791      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 38, 172, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 19, 86, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 17, 84, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 8, 42, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 21504)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     2,752,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,771,585 (10.57 MB)

 Trainable params: 2,771,585 (10.57 MB)

 Non-trainable params: 0 (0.00 B)

In [22]:
gender_model.fit(
    X_train, y_gender_train,
    validation_data=(X_test, y_gender_test),
    epochs=10,
    batch_size=32
)

Epoch 1/10


I0000 00:00:1768400692.380785     144 service.cc:152] XLA service 0x7daa84008390 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1768400692.380826     144 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1768400692.380830     144 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1768400692.721917     144 cuda_dnn.cc:529] Loaded cuDNN version 91002


 25/200 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.5541 - loss: 12.0345

I0000 00:00:1768400695.428352     144 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.7706 - loss: 3.0740 - val_accuracy: 0.9081 - val_loss: 0.2480
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9178 - loss: 0.2456 - val_accuracy: 0.9175 - val_loss: 0.2371
Epoch 3/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9346 - loss: 0.2020 - val_accuracy: 0.9269 - val_loss: 0.2031
Epoch 4/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9459 - loss: 0.1621 - val_accuracy: 0.9369 - val_loss: 0.2023
Epoch 5/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9500 - loss: 0.1441 - val_accuracy: 0.9319 - val_loss: 0.2264
Epoch 6/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9565 - loss: 0.1231 - val_accuracy: 0.9337 - val_loss: 0.2116
Epoch 7/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9664 - loss: 0.0922 - val_accuracy: 0.9356 - val_loss: 0.2262
Epoch 8/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9668 - loss: 0.0841 - val_accuracy: 0.9287 - val

In [24]:
age_model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=X_train.shape[1:]),
    MaxPooling2D((2,2)),
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1)  # regression output
])

age_model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

age_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_4 (Conv2D)               │ (None, 38, 172, 32)    │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 19, 86, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 17, 84, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 8, 42, 64)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 21504)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 128)            │     2,752,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,771,585 (10.57 MB)

 Trainable params: 2,771,585 (10.57 MB)

 Non-trainable params: 0 (0.00 B)

In [25]:
age_model.fit(
    X_train, y_age_train,
    validation_data=(X_test, y_age_test),
    epochs=10,
    batch_size=32
)

Epoch 1/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - loss: 114.9978 - mae: 3.6409 - val_loss: 0.0900 - val_mae: 0.1715
Epoch 2/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.1249 - mae: 0.2201 - val_loss: 0.0792 - val_mae: 0.1413
Epoch 3/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0850 - mae: 0.1472 - val_loss: 0.0787 - val_mae: 0.1273
Epoch 4/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0796 - mae: 0.1362 - val_loss: 0.0778 - val_mae: 0.1377
Epoch 5/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0870 - mae: 0.1425 - val_loss: 0.0770 - val_mae: 0.1488
Epoch 6/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0773 - mae: 0.1416 - val_loss: 0.0764 - val_mae: 0.1342
Epoch 7/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0782 - mae: 0.1371 - val_loss: 0.0786 - val_mae: 0.1262
Epoch 8/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.0825 - mae: 0.1389 - val_loss: 0.0772 - val_mae: 0.1450
Epoch 9/10
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step -

In [26]:
gender_model.save("voice_gender_model.h5")
age_model.save("voice_age_model.h5")